# AGGRESSIVE

# MURIL

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report

# ✅ Load datasets
train_df = pd.read_csv("/content/ag-train-binary-balanced.csv")
val_df = pd.read_csv("/content/ag-val-binary-balanced.csv")
test_df = pd.read_csv("/content/ag-test-binary-balanced.csv")

train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Load MuRIL
model_name = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ✅ Tokenize
def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='tf')

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Custom Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        # hidden_states shape = (batch_size, seq_len, hidden_dim)
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)  # (batch_size, seq_len, 1)
        attention_weights = tf.nn.softmax(score, axis=1)  # (batch_size, seq_len, 1)
        context_vector = attention_weights * hidden_states  # shape: (batch_size, seq_len, hidden_dim)
        context_vector = tf.reduce_sum(context_vector, axis=1)  # shape: (batch_size, hidden_dim)
        return context_vector

# ✅ Wrap HuggingFace MuRIL inside Keras Layer
class MuRILWrapper(tf.keras.layers.Layer):
    def __init__(self, model_name):
        super().__init__()
        self.muril = TFAutoModel.from_pretrained(model_name)

    def call(self, inputs):
        output = self.muril(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state  # shape: (batch_size, seq_len, hidden_size)

# ✅ Build model using Functional API
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

# Last hidden states from MuRIL
last_hidden_states = MuRILWrapper(model_name)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

# Apply Attention
context_vector = AttentionLayer()(last_hidden_states)

# Dense Layers
x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# ✅ Train
model.fit(train_dataset, validation_data=val_dataset, epochs=25)

# ✅ Predict + Report
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_binary = (preds.flatten() > 0.5).astype(int)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_binary, target_names=["NoAG", "AG"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


Some layers from the model checkpoint at google/muril-base-cased were not used when initializing TFBertModel: ['mlm___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFBertModel were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['bert/pooler/dense/kernel:0', 'bert/pooler/dense/bias:0']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 46s 181ms/step - accuracy: 0.5030 - loss: 0.6929 - val_accuracy: 0.5016 - val_loss: 0.6916
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 28s 130ms/step - accuracy: 0.5361 - loss: 0.6913 - val_accuracy: 0.5224 - val_loss: 0.6901
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 129ms/step - accuracy: 0.5745 - loss: 0.6899 - val_accuracy: 0.5737 - val_loss: 0.6885
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 137ms/step - accuracy: 0.6237 - loss: 0.6883 - val_accuracy: 0.6186 - val_loss: 0.6868
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 22s 132ms/step - accuracy: 0.6646 - loss: 0.6867 - val_accuracy: 0.6747 - val_loss: 0.6852
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 138ms/step - accuracy: 0.6899 - loss: 0.6849 - val_accuracy: 0.7115 - val_loss: 0.6835
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 135ms/step - accuracy: 0.7291 - loss: 0.6833 - val_accuracy: 0.7436 - val_loss: 0.6817
Epoch 8/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 40s 129ms/step - accuracy: 0.7503 - loss: 0

In [ ]:
!pip install --upgrade tensorflow transformers


In [ ]:
#%%time

import os
import re
import json
import nltk
import string
import random
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
import unicodedata
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

# TensorFlow & Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Sklearn utilities
from sklearn.utils import class_weight
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, roc_curve, precision_recall_curve
)
from sklearn.model_selection import train_test_split

# NLTK utilities
nltk.download('punkt')
from nltk.tokenize import word_tokenize, ToktokTokenizer
from nltk.corpus import stopwords

# Suppress warnings
warnings.filterwarnings('ignore')

# If using Jupyter Notebook only:
# %matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


# indicbert

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report

# ✅ Load datasets
train_df = pd.read_csv("/content/ag-train-binary-balanced.csv")
val_df = pd.read_csv("/content/ag-val-binary-balanced.csv")
test_df = pd.read_csv("/content/ag-test-binary-balanced.csv")

train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Load IndicBERT
model_name = "ai4bharat/indic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tf_model = TFAutoModel.from_pretrained(model_name, from_pt=True)

# ✅ Tokenize
def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='tf')

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Convert to tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Custom Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ Wrap IndicBERT inside Keras Layer
class IndicBERTWrapper(tf.keras.layers.Layer):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def call(self, inputs):
        output = self.model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state  # shape: (batch_size, seq_len, hidden_size)

# ✅ Build model using Functional API
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = IndicBERTWrapper(tf_model)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)

x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# ✅ Train
model.fit(train_dataset, validation_data=val_dataset, epochs=25)

# ✅ Evaluation Report
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_binary = (preds.flatten() > 0.5).astype(int)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_binary, target_names=["Non-Aggressive", "Aggressive"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFAlbertModel: ['predictions.dense.weight', 'predictions.LayerNorm.weight', 'predictions.dense.bias', 'predictions.bias', 'sop_classifier.classifier.bias', 'predictions.decoder.bias', 'predictions.decoder.weight', 'predictions.LayerNorm.bias', 'sop_classifier.classifier.weight']
- This IS expected if you are initializing TFAlbertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFAlbertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFAlbertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFAlbertModel

Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 52s 194ms/step - accuracy: 0.5474 - loss: 0.7004 - val_accuracy: 0.6907 - val_loss: 0.6418
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 126ms/step - accuracy: 0.6590 - loss: 0.6384 - val_accuracy: 0.7099 - val_loss: 0.6082
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 22s 129ms/step - accuracy: 0.6808 - loss: 0.6088 - val_accuracy: 0.7099 - val_loss: 0.5907
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 22s 130ms/step - accuracy: 0.6776 - loss: 0.6029 - val_accuracy: 0.7147 - val_loss: 0.5803
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 22s 128ms/step - accuracy: 0.6877 - loss: 0.5932 - val_accuracy: 0.7179 - val_loss: 0.5728
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 22s 128ms/step - accuracy: 0.6887 - loss: 0.5875 - val_accuracy: 0.7308 - val_loss: 0.5661
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 128ms/step - accuracy: 0.6991 - loss: 0.5789 - val_accuracy: 0.7324 - val_loss: 0.5620
Epoch 8/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 130ms/step - accuracy: 0.7052 - loss: 0

# mbert

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report

# ✅ Load datasets
train_df = pd.read_csv("/content/ag-train-binary-balanced.csv")
val_df = pd.read_csv("/content/ag-val-binary-balanced.csv")
test_df = pd.read_csv("/content/ag-test-binary-balanced.csv")

train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Load mBERT
mbert_model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(mbert_model_name)
tf_model = TFAutoModel.from_pretrained(mbert_model_name)

# ✅ Tokenize function
def tokenize(texts):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors='tf'
    )

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Convert to tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Custom Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ Wrap mBERT model inside Keras Layer
class MBERTWrapper(tf.keras.layers.Layer):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def call(self, inputs):
        output = self.model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state  # shape: (batch_size, seq_len, hidden_size)

# ✅ Build the model using Functional API
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = MBERTWrapper(tf_model)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)

x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(2e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ✅ Train
model.fit(train_dataset, validation_data=val_dataset, epochs=25)

# ✅ Evaluate
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_binary = (preds.flatten() > 0.5).astype(int)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_binary, target_names=["Non-Aggressive", "Aggressive"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 53s 201ms/step - accuracy: 0.5316 - loss: 0.7247 - val_accuracy: 0.6779 - val_loss: 0.6385
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 22s 131ms/step - accuracy: 0.6774 - loss: 0.6288 - val_accuracy: 0.7484 - val_loss: 0.5837
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 22s 130ms/step - accuracy: 0.7405 - loss: 0.5794 - val_accuracy: 0.7644 - val_loss: 0.5454
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 22s 131ms/step - accuracy: 0.7654 - loss: 0.5411 - val_accuracy: 0.7788 - val_loss: 0.5160
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 22s 132ms/step - accuracy: 0.7936 - loss: 0.5110 - val_accuracy: 0.7901 - val_loss: 0.4930
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 131ms/step - accuracy: 0.7923 - loss: 0.4893 - val_accuracy: 0.7981 - val_loss: 0.4744
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 22s 133ms/step - accuracy: 0.8058 - loss: 0.4730 - val_accuracy: 0.8093 - val_loss: 0.4579
Epoch 8/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 40s 129ms/step - accuracy: 0.8046 - loss: 0

# bangla bert

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report

# ✅ Load datasets
train_df = pd.read_csv("/content/ag-train-binary-balanced.csv")
val_df = pd.read_csv("/content/ag-val-binary-balanced.csv")
test_df = pd.read_csv("/content/ag-test-binary-balanced.csv")

train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Load BanglaBERT
banglabert_model_name = "csebuetnlp/banglabert_large"
tokenizer = AutoTokenizer.from_pretrained(banglabert_model_name)
banglabert_model = TFAutoModel.from_pretrained(banglabert_model_name, from_pt=True)

# ✅ Tokenize function
def tokenize(texts):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors='tf'
    )

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Create tf.data.Datasets
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Custom Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ Wrap BanglaBERT in a Keras Layer
class BanglaBERTWrapper(tf.keras.layers.Layer):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def call(self, inputs):
        output = self.model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state  # shape: (batch_size, seq_len, hidden_size)

# ✅ Build the model using Functional API
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = BanglaBERTWrapper(banglabert_model)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)

x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(2e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ✅ Train
model.fit(train_dataset, validation_data=val_dataset, epochs=25)

# ✅ Evaluate
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_binary = (preds.flatten() > 0.5).astype(int)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_binary, target_names=["Non-Aggressive", "Aggressive"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFElectraModel: ['discriminator_predictions.dense.weight', 'electra.embeddings.position_ids', 'discriminator_predictions.dense.bias', 'discriminator_predictions.dense_prediction.weight', 'discriminator_predictions.dense_prediction.bias']
- This IS expected if you are initializing TFElectraModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFElectraModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFElectraModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFElectraModel for predictions without further train

Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 129s 573ms/step - accuracy: 0.5352 - loss: 0.7621 - val_accuracy: 0.6795 - val_loss: 0.6326
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 112s 456ms/step - accuracy: 0.6206 - loss: 0.6432 - val_accuracy: 0.7196 - val_loss: 0.5895
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 76s 450ms/step - accuracy: 0.6690 - loss: 0.6146 - val_accuracy: 0.7276 - val_loss: 0.5649
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 76s 451ms/step - accuracy: 0.7002 - loss: 0.5979 - val_accuracy: 0.7388 - val_loss: 0.5467
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 76s 451ms/step - accuracy: 0.7048 - loss: 0.5786 - val_accuracy: 0.7404 - val_loss: 0.5339
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 76s 451ms/step - accuracy: 0.7014 - loss: 0.5680 - val_accuracy: 0.7452 - val_loss: 0.5238
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 76s 452ms/step - accuracy: 0.7131 - loss: 0.5610 - val_accuracy: 0.7516 - val_loss: 0.5159
Epoch 8/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 82s 452ms/step - accuracy: 0.7191 - loss:

# **EMOTION**

# MURIL

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ✅ Load dataset
train_df = pd.read_csv("/content/em-train-balanced.csv")
val_df = pd.read_csv("/content/em-train-balanced.csv")
test_df = pd.read_csv("/content/em-test-balanced.csv")

# ✅ Extract texts and labels
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Tokenizer
model_name = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='tf')

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Prepare tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ MuRIL Wrapper
class MuRILWrapper(tf.keras.layers.Layer):
    def __init__(self, model_name):
        super().__init__()
        self.muril = TFAutoModel.from_pretrained(model_name)

    def call(self, inputs):
        output = self.muril(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state

# ✅ Model Architecture
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = MuRILWrapper(model_name)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)
x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(6, activation='softmax')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile model
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# ✅ Compute class weights to upweight label=2 (direct violence)
class_weights = compute_class_weight(class_weight='balanced',
                                     classes=np.unique(train_labels),
                                     y=train_labels)
class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}
print("Class Weights:", class_weights_dict)

# ✅ Train model
model.fit(train_dataset,
          validation_data=val_dataset,
          epochs=25,
          class_weight=class_weights_dict)

# ✅ Evaluate
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_class = np.argmax(preds, axis=1)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_class, target_names=["joy", "sad", "surprise","disgust","anger","fear"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


Some layers from the model checkpoint at google/muril-base-cased were not used when initializing TFBertModel: ['mlm___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFBertModel were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['bert/pooler/dense/kernel:0', 'bert/pooler/dense/bias:0']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Class Weights: {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0), 3: np.float64(1.0), 4: np.float64(1.0), 5: np.float64(1.0)}
Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 94s 446ms/step - accuracy: 0.1629 - loss: 1.7924 - val_accuracy: 0.1667 - val_loss: 1.7914
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 102s 253ms/step - accuracy: 0.1579 - loss: 1.7917 - val_accuracy: 0.1667 - val_loss: 1.7906
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 80s 244ms/step - accuracy: 0.1794 - loss: 1.7905 - val_accuracy: 0.1689 - val_loss: 1.7897
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 40s 238ms/step - accuracy: 0.1996 - loss: 1.7897 - val_accuracy: 0.1852 - val_loss: 1.7889
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 241ms/step - accuracy: 0.2011 - loss: 1.7891 - val_accuracy: 0.2126 - val_loss: 1.7880
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 242ms/step - accuracy: 0.2268 - loss: 1.7883 - val_accuracy: 0.2359 - val_loss: 1.7872
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 242ms/step - accuracy: 0.2576 - l

# XLMROBERTA

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ✅ Load dataset
train_df = pd.read_csv("/content/em-train-balanced.csv")
val_df = pd.read_csv("/content/em-train-balanced.csv")
test_df = pd.read_csv("/content/em-test-balanced.csv")


# ✅ Extract texts and labels
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Load XLM-RoBERTa tokenizer
model_name = "xlm-roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='tf')

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Prepare tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Custom Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ XLM-R Wrapper
class XLMRobertaWrapper(tf.keras.layers.Layer):
    def __init__(self, model_name):
        super().__init__()
        self.model = TFAutoModel.from_pretrained(model_name)

    def call(self, inputs):
        output = self.model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state

# ✅ Build Model
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = XLMRobertaWrapper(model_name)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)
x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(6, activation='softmax')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# ✅ Compute Class Weights
class_weights = compute_class_weight(class_weight='balanced',
                                     classes=np.unique(train_labels),
                                     y=train_labels)
class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}
print("Class Weights:", class_weights_dict)

# ✅ Train Model
model.fit(train_dataset,
          validation_data=val_dataset,
          epochs=25,
          class_weight=class_weights_dict)

# ✅ Evaluation Report
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_class = np.argmax(preds, axis=1)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_class, target_names=["joy", "sad", "surprise","disgust","anger","fear"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFXLMRobertaModel: ['lm_head.bias', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.weight', 'lm_head.layer_norm.bias']
- This IS expected if you are initializing TFXLMRobertaModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFXLMRobertaModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFXLMRobertaModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFXLMRobertaModel for predictions without further training.


Class Weights: {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0), 3: np.float64(1.0), 4: np.float64(1.0), 5: np.float64(1.0)}
Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 182s 874ms/step - accuracy: 0.1651 - loss: 2.3334 - val_accuracy: 0.1893 - val_loss: 2.0007
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 156s 667ms/step - accuracy: 0.1678 - loss: 2.0168 - val_accuracy: 0.1922 - val_loss: 1.8467
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 112s 665ms/step - accuracy: 0.1789 - loss: 1.8890 - val_accuracy: 0.2070 - val_loss: 1.7991
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 112s 664ms/step - accuracy: 0.1807 - loss: 1.8514 - val_accuracy: 0.2244 - val_loss: 1.7883
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 143s 668ms/step - accuracy: 0.1802 - loss: 1.8289 - val_accuracy: 0.2441 - val_loss: 1.7828
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 112s 664ms/step - accuracy: 0.1731 - loss: 1.8322 - val_accuracy: 0.2556 - val_loss: 1.7777
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 112s 662ms/step - accuracy: 0.17

# MBERT

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ✅ Load dataset
train_df = pd.read_csv("/content/em-train-balanced.csv")
val_df = pd.read_csv("/content/em-train-balanced.csv")
test_df = pd.read_csv("/content/em-test-balanced.csv")

# ✅ Extract texts and labels
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Tokenizer for mBERT
mbert_model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(mbert_model_name)

def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='tf')

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Prepare tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ mBERT Wrapper
class mBERTWrapper(tf.keras.layers.Layer):
    def __init__(self, model_name):
        super().__init__()
        self.bert = TFAutoModel.from_pretrained(model_name)

    def call(self, inputs):
        output = self.bert(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state

# ✅ Model Architecture
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = mBERTWrapper(mbert_model_name)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)
x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(6, activation='softmax')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile model
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# ✅ Compute class weights
class_weights = compute_class_weight(class_weight='balanced',
                                     classes=np.unique(train_labels),
                                     y=train_labels)
class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}
print("Class Weights:", class_weights_dict)

# ✅ Train model
model.fit(train_dataset,
          validation_data=val_dataset,
          epochs=25,
          class_weight=class_weights_dict)

# ✅ Evaluation Report
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_class = np.argmax(preds, axis=1)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_class, target_names=["joy", "sad", "surprise","disgust","anger","fear"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

Class Weights: {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0), 3: np.float64(1.0), 4: np.float64(1.0), 5: np.float64(1.0)}
Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 71s 325ms/step - accuracy: 0.1902 - loss: 1.8318 - val_accuracy: 0.2015 - val_loss: 1.7827
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 64s 248ms/step - accuracy: 0.1889 - loss: 1.7845 - val_accuracy: 0.2341 - val_loss: 1.7612
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 40s 238ms/step - accuracy: 0.2306 - loss: 1.7679 - val_accuracy: 0.2607 - val_loss: 1.7423
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 243ms/step - accuracy: 0.2279 - loss: 1.7547 - val_accuracy: 0.2922 - val_loss: 1.7248
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 242ms/step - accuracy: 0.2653 - loss: 1.7313 - val_accuracy: 0.3211 - val_loss: 1.7082
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 242ms/step - accuracy: 0.2806 - loss: 1.7123 - val_accuracy: 0.3385 - val_loss: 1.6921
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 243ms/step - accuracy: 0.2960 - lo

# INDICBERT

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ✅ Load dataset
train_df = pd.read_csv("/content/em-train-balanced.csv")
val_df = pd.read_csv("/content/em-train-balanced.csv")
test_df = pd.read_csv("/content/em-test-balanced.csv")

# ✅ Extract texts and labels
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Tokenizer for IndicBERT
model_name = "ai4bharat/indic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='tf')

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Prepare tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ IndicBERT Wrapper (instead of MuRIL)
class IndicBERTWrapper(tf.keras.layers.Layer):
    def __init__(self, model_name):
        super().__init__()
        self.indicbert = TFAutoModel.from_pretrained(model_name, from_pt=True)

    def call(self, inputs):
        output = self.indicbert(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state

# ✅ Model Architecture
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = IndicBERTWrapper(model_name)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)
x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(6, activation='softmax')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile model
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# ✅ Compute class weights
class_weights = compute_class_weight(class_weight='balanced',
                                     classes=np.unique(train_labels),
                                     y=train_labels)
class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}
print("Class Weights:", class_weights_dict)

# ✅ Train model
model.fit(train_dataset,
          validation_data=val_dataset,
          epochs=25,
          class_weight=class_weights_dict)

# ✅ Evaluation
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_class = np.argmax(preds, axis=1)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_class, target_names=["joy", "sad", "surprise","disgust","anger","fear"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


pytorch_model.bin:   0%|          | 0.00/135M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFAlbertModel: ['predictions.dense.weight', 'predictions.decoder.bias', 'predictions.LayerNorm.weight', 'predictions.LayerNorm.bias', 'predictions.bias', 'sop_classifier.classifier.weight', 'predictions.decoder.weight', 'predictions.dense.bias', 'sop_classifier.classifier.bias']
- This IS expected if you are initializing TFAlbertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFAlbertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFA

Class Weights: {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0), 3: np.float64(1.0), 4: np.float64(1.0), 5: np.float64(1.0)}
Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 63s 270ms/step - accuracy: 0.1608 - loss: 1.8532 - val_accuracy: 0.1793 - val_loss: 1.7880
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 36s 212ms/step - accuracy: 0.1728 - loss: 1.7964 - val_accuracy: 0.2078 - val_loss: 1.7775
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 213ms/step - accuracy: 0.1990 - loss: 1.7817 - val_accuracy: 0.2256 - val_loss: 1.7689
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 43s 227ms/step - accuracy: 0.2051 - loss: 1.7818 - val_accuracy: 0.2433 - val_loss: 1.7612
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 39s 213ms/step - accuracy: 0.2147 - loss: 1.7719 - val_accuracy: 0.2589 - val_loss: 1.7539
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 213ms/step - accuracy: 0.2473 - loss: 1.7568 - val_accuracy: 0.2685 - val_loss: 1.7476
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 213ms/step - accuracy: 0.2435 - lo

# BANGLA BERT

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ✅ Load dataset
train_df = pd.read_csv("/content/em-train-balanced.csv")
val_df = pd.read_csv("/content/em-train-balanced.csv")
test_df = pd.read_csv("/content/em-test-balanced.csv")

# ✅ Extract texts and labels
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Tokenizer for BanglaBERT
banglabert_model_name = "csebuetnlp/banglabert_large"
tokenizer = AutoTokenizer.from_pretrained(banglabert_model_name)

def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='tf')

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Prepare tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ BanglaBERT Wrapper
class BanglaBERTWrapper(tf.keras.layers.Layer):
    def __init__(self, model_name):
        super().__init__()
        self.banglabert = TFAutoModel.from_pretrained(model_name, from_pt=True)

    def call(self, inputs):
        output = self.banglabert(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state

# ✅ Model Architecture
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = BanglaBERTWrapper(banglabert_model_name)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)
x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(6, activation='softmax')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile model
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# ✅ Compute class weights
class_weights = compute_class_weight(class_weight='balanced',
                                     classes=np.unique(train_labels),
                                     y=train_labels)
class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}
print("Class Weights:", class_weights_dict)

# ✅ Train model
model.fit(train_dataset,
          validation_data=val_dataset,
          epochs=25,
          class_weight=class_weights_dict)

# ✅ Evaluation Report
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_class = np.argmax(preds, axis=1)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_class, target_names=["joy", "sad", "surprise","disgust","anger","fear"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFElectraModel: ['discriminator_predictions.dense.bias', 'discriminator_predictions.dense_prediction.weight', 'discriminator_predictions.dense_prediction.bias', 'discriminator_predictions.dense.weight', 'electra.embeddings.position_ids']
- This IS expected if you are initializing TFElectraModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFElectraModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFElectraModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFElectraModel for predictions without further train

Class Weights: {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0), 3: np.float64(1.0), 4: np.float64(1.0), 5: np.float64(1.0)}
Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 190s 937ms/step - accuracy: 0.1566 - loss: 1.8970 - val_accuracy: 0.1478 - val_loss: 1.8135
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 136s 809ms/step - accuracy: 0.1560 - loss: 1.8459 - val_accuracy: 0.1596 - val_loss: 1.8006
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 156s 893ms/step - accuracy: 0.1657 - loss: 1.8263 - val_accuracy: 0.1837 - val_loss: 1.7892
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 188s 812ms/step - accuracy: 0.1592 - loss: 1.8300 - val_accuracy: 0.2037 - val_loss: 1.7785
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 141s 807ms/step - accuracy: 0.1864 - loss: 1.8263 - val_accuracy: 0.2156 - val_loss: 1.7704
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 142s 809ms/step - accuracy: 0.2048 - loss: 1.7924 - val_accuracy: 0.2281 - val_loss: 1.7629
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 142s 808ms/step - accuracy: 0.20